In [7]:
import tensorflow
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [14]:
#Load the trained model, scale pickle file and one hot encoded model

model = load_model('model.h5')

#load the encoder and scaler file 
with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)

with open('onehot_encoder_geo.pkl','rb') as file:
    label_encoder_geo = pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)

In [15]:
#example input data

input_data = {
    'CreditScore' : 600,
    'Geography' : 'France',
    'Gender' : 'Male',
    'Age' : 40,
    'Tenure' : 3,
    'Balance' : 60000,
    'NumofProducts' : 2,
    'HasCrCard' : 1,
    'IsActiveMember' : 1,
    'EstimatedSalart' : 50000
}

#Gender would be label to 0 and 1, geography - ohe and that be defined, salary dropout

In [22]:
#For geography:

geo_encoded = label_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=label_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

/opt/miniconda3/envs/ai-env/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [23]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumofProducts,HasCrCard,IsActiveMember,EstimatedSalart
0,600,France,Male,40,3,60000,2,1,1,50000


In [24]:
#Encode categorical variables:

input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumofProducts,HasCrCard,IsActiveMember,EstimatedSalart
0,600,France,1,40,3,60000,2,1,1,50000


In [26]:
#Concatenation with ohe 
input_df = pd.concat([input_df.drop("Geography",axis=1), geo_encoded_df], axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumofProducts,HasCrCard,IsActiveMember,EstimatedSalart,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [28]:
# Fix missing columns
for col in scaler.feature_names_in_:
    if col not in input_df.columns:
        input_df[col] = 0

# Fix order
input_df = input_df[scaler.feature_names_in_]

# Scale
input_scaled = scaler.transform(input_df)

In [29]:
#Scaling the input data
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
        -2.6418115 ,  0.64920267,  0.97481699, -1.74616572,  1.00150113,
        -0.57946723, -0.57638802]])

In [31]:
prediction = model.predict(input_scaled)
prediction


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step


array([[0.35278514]], dtype=float32)

In [33]:
prediction_proba = prediction[0][0]
prediction_proba

np.float32(0.35278514)

In [34]:
if prediction_proba > 0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is likely to not churn")

The customer is likely to not churn
